# 04 Time, place, person — notebook

The line list is clean (chapter 03). Now Maya describes the outbreak the way every field
epidemiologist starts: **when** did people fall ill, **where** do they live, and **who**
are they? Every number you see here is computed from the synthetic Santa Luz data —
Santa Luz City, Typhoon Amihan and all the cases are invented for teaching.

You will make: a table of case classifications, an epidemic curve with the typhoon
marked, attack rates per 10,000 by band and barangay, an age–sex chart and the
case-fatality rate.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 · Load the clean line list and the denominators

Two files: the **cases** (`santa_luz_linelist.csv`, one row per case) and the
**population** of each barangay (`santa_luz_barangays.csv`). We tell pandas which
columns are dates so it reads them as real dates, not text.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

DATE_COLUMNS = ["date_onset", "date_consult", "date_report", "date_lab_confirmed"]
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv", parse_dates=DATE_COLUMNS)
barangays = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")

LANDFALL = pd.Timestamp("2026-07-18")  # Typhoon Amihan
BANDS = ["riverside", "coastal", "upland"]

print(f"{len(ll)} cases | {len(barangays)} barangays | {barangays['population'].sum():,} residents")
ll["disease"].value_counts()

## Step 2 · Case classification

Every row was classified with a case definition at the 30 September data freeze:
**suspect** (meets the clinical description), **probable** (plus a positive rapid
test), **confirmed** (plus a positive laboratory result). A cross-table counts each
combination; `margins=True` adds totals.

In [ ]:
classes = pd.crosstab(ll["disease"], ll["case_class"], margins=True, margins_name="Total")
classes[["confirmed", "probable", "suspect", "Total"]]

## Step 3 · The epidemic curve (time)

> **Ask Tala:** "My DataFrame `lepto` has a datetime column `date_onset` (made-up rows:
> `2026-07-24`, `2026-07-24`, `2026-07-27`). Draw an epidemic curve: one bar per day from
> 15 July to 31 August 2026, **including days with zero cases**, no gaps between bars.
> Mark 18 July with a red dashed line labelled 'Typhoon Amihan landfall', shade the flood
> days 18–20 July and the clean-up 21 July – 2 August."

The important trick is `reindex(..., fill_value=0)`: `value_counts()` lists only days
that *have* cases, so we lay those counts onto a full calendar.

In [ ]:
lepto = ll[ll["disease"] == "leptospirosis"]
days = pd.date_range("2026-07-15", "2026-08-31", freq="D")
daily = lepto["date_onset"].value_counts().reindex(days, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axvspan(pd.Timestamp("2026-07-17 12:00"), pd.Timestamp("2026-07-20 12:00"),
           color="#4FA3D9", alpha=0.30, label="Flood (18–20 Jul)")
ax.axvspan(pd.Timestamp("2026-07-20 12:00"), pd.Timestamp("2026-08-02 12:00"),
           color="#FCD116", alpha=0.25, label="Clean-up (21 Jul – ~2 Aug)")
ax.bar(daily.index, daily.values, width=1.0, color="#0038A8", edgecolor="white", linewidth=0.6)
ax.axvline(LANDFALL, color="#CE1126", linestyle="--", linewidth=2,
           label="Typhoon Amihan landfall (18 Jul)")
ax.set_title("Leptospirosis cases by date of onset, Santa Luz City (synthetic data)")
ax.set_xlabel("Date of onset (2026)")
ax.set_ylabel("Number of cases")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**Check it.** The bars must add up to every leptospirosis case **with** an onset date
(2 have none — chapter 03, rule C4). The weekly totals must match the outbreak report.
And find the peak day.

In [ ]:
missing_onset = lepto["date_onset"].isna().sum()
assert daily.sum() == len(lepto) - missing_onset
print(f"{daily.sum()} bars + {missing_onset} without onset = {len(lepto)} cases")

weekly = lepto["date_onset"].dt.isocalendar().week.value_counts().sort_index()
weekly.index = weekly.index.astype(int)
print("Cases by ISO onset week:", weekly.to_dict())
assert weekly.to_dict() == {30: 35, 31: 108, 32: 68, 33: 15, 34: 2}
print("Peak day:", daily.idxmax().date(), "with", daily.max(), "cases")

## Step 4 · What does the shape say?

Read the curve like a detective:

- **The first onset is 22 July**, four days after landfall (the enhanced line list
  collects onsets from 20 July). People needed time to fall ill: the **incubation
  period**, the time from infection to the first symptoms.
- **One wave**: rising fast, staying high for about two weeks (24 July – 6 August), then
  fading by mid-August. No second, third or fourth wave.
- Leptospirosis does not usually spread from person to person, and the wave lines up
  with the flood and the clean-up, when people kept wading in water that may have been
  contaminated.

That is the signature of a **common source** that people were exposed to over a period
of days — a **continuing common source** — not a single moment (a *point source* would
give an even sharper single peak) and not a *propagated* outbreak (successive waves, each
one incubation period apart). Counting days from landfall makes the reasoning concrete.

For comparison we also draw the **dengue** curve by week: dengue is carried by *Aedes*
mosquitoes that breed in standing water, so it climbs later and keeps climbing for weeks.

In [ ]:
days_after = (lepto["date_onset"] - LANDFALL).dt.days.dropna().astype(int)
print("Days from landfall to onset: first", days_after.min(), "| median", int(days_after.median()),
      "| last", days_after.max())

dengue = ll[ll["disease"] == "dengue"]
dengue_weeks = dengue["date_onset"].dt.isocalendar().week.value_counts().reindex(range(30, 40), fill_value=0)
print("Dengue cases by ISO onset week:", dengue_weeks.to_dict())

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([f"W{w}" for w in dengue_weeks.index], dengue_weeks.values, width=0.95, color="#00737D")
ax.set_title("Dengue cases by week of onset, 2026 (synthetic; W39 still incomplete)")
ax.set_xlabel("ISO week of onset")
ax.set_ylabel("Number of cases")
plt.tight_layout()
plt.show()

## Step 5 · Tala got it wrong: "attack rate" without a population

> **Ask Tala:** "Compute the dengue attack rate by `band`."

Tala's answer runs and looks tidy:

In [ ]:
# Tala's first answer: share of CASES in each band -- NOT an attack rate
wrong = (dengue["band"].value_counts(normalize=True) * 100).round(1)
print("Tala's 'attack rate (%)':")
print(wrong)
riverside_share = (lepto["band"] == "riverside").mean() * 100
print(f"Same mistake for leptospirosis: {riverside_share:.0f}% of cases are riverside residents")

These numbers add up to 100 %. That is the clue: they say **where the cases are**, not
**how risky it is to live there**. The riverside has the most dengue cases simply because
it has the most people (96,064 of 186,000). An **attack rate** is

$$\text{attack rate} = \frac{\text{new cases in a group}}{\text{people at risk in that group}}$$

and for a city we express it **per 10,000 residents**. The denominator comes from the
barangay file, not from the line list.

In [ ]:
population = barangays.groupby("band")["population"].sum()
cases = pd.crosstab(ll["band"], ll["disease"])

attack = cases.div(population, axis=0) * 10_000
attack.loc["city"] = cases.sum() / population.sum() * 10_000
attack = attack.loc[BANDS + ["city"], ["leptospirosis", "dengue"]].round(1)
print("Attack rate per 10,000 residents")
attack

**Check it** by hand for one cell, and compare the two answers. For dengue, Tala's
version put the riverside first; the real attack rate says the **coastal** band carries
the highest risk.

In [ ]:
riverside_lepto = (ll["disease"].eq("leptospirosis") & ll["band"].eq("riverside")).sum()
print(f"Riverside leptospirosis: {riverside_lepto} / {population['riverside']:,} x 10,000 =",
      round(riverside_lepto / population["riverside"] * 10_000, 1))
assert attack.loc["riverside", "leptospirosis"] == 19.7
assert attack.loc["coastal", "dengue"] == 29.6
assert attack["dengue"].drop("city").idxmax() == "coastal" != wrong.idxmax()

## Step 6 · Place: attack rate by barangay

Now the same idea for each of the 24 barangays. We **start from the barangay table**
and join the case counts onto it, so a barangay with no cases (upland Bukid) still
appears — with a rate of 0, not missing.

In [ ]:
counts = ll.groupby(["barangay_id", "disease"]).size().unstack(fill_value=0)
place = barangays.set_index("barangay_id")[["barangay", "band", "population"]].join(counts).fillna(0)
place[["leptospirosis", "dengue"]] = place[["leptospirosis", "dengue"]].astype(int)
for disease in ["leptospirosis", "dengue"]:
    place[f"ar_{disease}"] = (place[disease] / place["population"] * 10_000).round(1)

print(len(place), "barangays;", (place["leptospirosis"] + place["dengue"] == 0).sum(), "with no cases")
top = place.sort_values("ar_leptospirosis", ascending=False)
top[["barangay", "band", "leptospirosis", "population", "ar_leptospirosis"]].head(4)

A bar chart sorted by rate, coloured by band, shows the pattern at a glance (chapter 08
draws it as a map).

In [ ]:
BAND_COLOURS = {"riverside": "#0038A8", "coastal": "#00737D", "upland": "#2F7D32"}
fig, axes = plt.subplots(1, 2, figsize=(11, 6.5), sharey=False)
for ax, disease in zip(axes, ["leptospirosis", "dengue"]):
    data = place.sort_values(f"ar_{disease}")
    ax.barh(data["barangay"], data[f"ar_{disease}"], color=data["band"].map(BAND_COLOURS))
    ax.set_title(disease.capitalize())
    ax.set_xlabel("Cases per 10,000 residents")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in BAND_COLOURS.values()]
fig.legend(handles, BAND_COLOURS.keys(), loc="lower center", ncol=3)
fig.suptitle("Attack rate by barangay, Santa Luz City (synthetic data)")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

print("Dengue, highest two:", place.nlargest(2, "ar_dengue")[["barangay", "ar_dengue"]].values.tolist())

## Step 7 · Person: age, sex and deaths

Who fell ill? A side-by-side **age–sex chart** (men to the left, women to the right) for
each disease, plus the median age and the share of men. Seven dengue cases have no age,
so we count them separately instead of letting them vanish.

Then the **case-fatality rate** (CFR): of the people who became cases, what share died?
Here the denominator *is* the cases — that is the definition, and the opposite of the
attack-rate mistake.

In [ ]:
AGE_GROUPS = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, disease in zip(axes, ["leptospirosis", "dengue"]):
    d = ll[ll["disease"] == disease]
    table = pd.crosstab(d["age_group"], d["sex"]).reindex(AGE_GROUPS, fill_value=0)
    ax.barh(AGE_GROUPS, -table["M"], color="#0038A8", label="Male")
    ax.barh(AGE_GROUPS, table["F"], color="#CE1126", label="Female")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{disease.capitalize()} (n = {len(d)})")
    ax.set_xlabel("Cases (male left, female right)")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{abs(int(x))}")
axes[0].set_ylabel("Age group (years)")
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

for disease in ["leptospirosis", "dengue"]:
    d = ll[ll["disease"] == disease]
    male = (d["sex"] == "M").sum()
    print(f"{disease:<14} median age {d['age'].median():.0f} | male {male} ({male / len(d):.0%})"
          f" | age missing {d['age'].isna().sum()}")

# Case-fatality rate: deaths / CASES (here the cases ARE the right denominator)
cfr = ll.groupby("disease").agg(cases=("case_id", "size"),
                                deaths=("outcome", lambda s: (s == "died").sum()),
                                hospitalised=("hospitalised", "sum"))
cfr["cfr_pct"] = (cfr["deaths"] / cfr["cases"] * 100).round(1)
cfr["hospitalised_pct"] = (cfr["hospitalised"] / cfr["cases"] * 100).round(1)
assert cfr.loc["leptospirosis", "deaths"] == 20 and cfr.loc["leptospirosis", "cfr_pct"] == 8.7
cfr

## Your turn

1. **Dengue epi curve by day.** Copy the leptospirosis curve cell and draw dengue by
   onset day from 15 July to 30 September. Does it look like one wave from one source?
   *Hint: change the filter and the `date_range`; leave out the clean-up shading.*
2. **Attack rate by sex.** The barangay file has no sex split, so assume half the
   population is male. Compute leptospirosis cases per 10,000 men and per 10,000 women.
   *Hint: `lepto["sex"].value_counts() / (186_000 / 2) * 10_000`, and say clearly in
   your note that the denominator is an assumption.*
3. **CFR by age group.** Is leptospirosis more deadly in older cases? *Hint:
   `lepto.groupby("age_group")["outcome"].apply(lambda s: (s == "died").mean())` — and
   print the number of cases in each group too, because a CFR from 9 cases is shaky.*